<div dir="rtl">

# פרק 8: סימולציות ומספרים רנדומליים

*איך מגרילים מספרים, ואיך משתמשים בהגרלה חוזרת כדי לענות על שאלות הסתברות ולדמות התפשטות של מחלה.*

**איך לעבוד עם המחברת:** הרץ כל תא קוד עם Shift+Enter. בתרגילים, כתוב את הפתרון בתא הריק, ורק אחר כך פתח את הפתרון בלחיצה כפולה על התא שמתחתיו. כדי לשמור את העבודה שלך: **File → Save a copy in GitHub** (או **Save a copy in Drive**).

</div>

In [ ]:
import matplotlib.pyplot as plt   # used for the graphs in this chapter

<div dir="rtl">

## המודול random

</div>

In [ ]:
import random
random.seed(42)    # makes the "random" numbers repeat every run

print(random.random())                         # float between 0 and 1
print(random.randint(1, 6))                    # whole number 1..6 (including 6)
print(random.choice(["A", "B", "AB", "O"]))    # one item from a list
print(round(random.gauss(120, 15), 1))         # normal distribution: mean 120, sd 15
print(random.sample(range(1, 21), 5))          # 5 different numbers, no repeats

<div dir="rtl">

**פלט צפוי:**

```text
0.6394267984578837
1
AB
120.3
[4, 18, 3, 14, 2]
```

> 💡 **מה זה seed**  
> מחשב לא באמת מגריל. הוא מחשב סדרה שנראית אקראית, מנקודת התחלה שנקראת seed. אותו seed תמיד נותן אותה סדרה. זה חשוב במחקר: מישהו אחר יכול להריץ את הקוד שלך ולקבל בדיוק את אותן תוצאות. בלי `seed` תקבל מספרים אחרים בכל הרצה.

`random.shuffle(my_list)` מערבב רשימה במקום. שימושי למשל לחלוקה אקראית של מטופלים לקבוצות במחקר.

## סימולציית מונטה קרלו

הרעיון: במקום לחשב הסתברות בנוסחה, מבצעים את הניסוי במחשב הרבה מאוד פעמים וסופרים. לפי חוק המספרים הגדולים, ככל שיש יותר חזרות, התוצאה מתקרבת להסתברות האמיתית.

דוגמה פשוטה: מה הסיכוי שסכום שתי קוביות יהיה 7?

</div>

In [ ]:
random.seed(1)
N = 100_000          # underscores just make big numbers readable
hits = 0
for _ in range(N):   # _ means "I don't need the loop variable"
    if random.randint(1, 6) + random.randint(1, 6) == 7:
        hits += 1

print(hits / N, 6 / 36)

<div dir="rtl">

**פלט צפוי:**

```text
0.16621 0.16666666666666666
```

הסימולציה (0.166) כמעט זהה לתשובה המדויקת (6/36).

## דוגמה רפואית: מה המשמעות של בדיקה חיובית?

מחלה שכיחה ב-1% מהאוכלוסייה. הבדיקה מזהה 90% מהחולים (רגישות) ונותנת תוצאה שלילית ל-95% מהבריאים (סגוליות). אדם קיבל תוצאה חיובית. מה הסיכוי שהוא באמת חולה?

רוב האנשים, כולל רופאים, עונים "בערך 90%". נבדוק בסימולציה:

</div>

In [ ]:
random.seed(0)
N = 100_000
prevalence, sensitivity, specificity = 0.01, 0.90, 0.95
true_pos = 0
false_pos = 0

for _ in range(N):
    sick = random.random() < prevalence            # True 1% of the time
    if sick:
        positive = random.random() < sensitivity   # 90% of sick test positive
    else:
        positive = random.random() > specificity   # 5% of healthy test positive
    if positive and sick:
        true_pos += 1
    elif positive:
        false_pos += 1

ppv = true_pos / (true_pos + false_pos)
print(f"Positive tests: {true_pos + false_pos}")
print(f"Chance a positive is really sick: {ppv:.1%}")

<div dir="rtl">

**פלט צפוי:**

```text
Positive tests: 5714
Chance a positive is really sick: 15.1%
```

רק כ-15%. הסיבה: יש הרבה יותר בריאים מחולים, ולכן 5% טעויות מתוך 99% בריאים הם יותר מ-90% מתוך 1% חולים. זה נקרא **ערך ניבוי חיובי** (PPV), והחישוב המדויק לפי משפט בייס נותן 15.4%. הסימולציה מתקרבת אליו בלי שהיינו צריכים לזכור את הנוסחה.

`random.random() < p` הוא הטריק המרכזי כאן: התנאי מתקיים בהסתברות p בדיוק.

## סימולציה של מגפה: מודל SIR

מודל SIR מחלק את האוכלוסייה לשלוש קבוצות: S רגישים (Susceptible), I נדבקים (Infected) ו-R מחלימים (Recovered). בכל יום חלק מהרגישים נדבקים, וחלק מהנדבקים מחלימים. β הוא קצב ההדבקה ו-γ קצב ההחלמה.

</div>

In [ ]:
S, I, R = 990, 10, 0
beta, gamma = 0.3, 0.1
N = S + I + R
S_list, I_list, R_list = [S], [I], [R]

for day in range(1, 161):
    new_inf = beta * S * I / N
    new_rec = gamma * I
    S -= new_inf
    I += new_inf - new_rec
    R += new_rec
    S_list.append(S)
    I_list.append(I)
    R_list.append(R)

peak = max(I_list)
print(f"Peak: {peak:.0f} infected on day {I_list.index(peak)}")

plt.plot(S_list, label="Susceptible")
plt.plot(I_list, label="Infected")
plt.plot(R_list, label="Recovered")
plt.xlabel("Day")
plt.ylabel("People")
plt.legend()
plt.show()

<div dir="rtl">

**פלט צפוי:**

```text
Peak: 316 infected on day 28
```

היחס β/γ נקרא R0 (מספר ההדבקה הבסיסי), וכאן הוא 3. נסה לשנות את beta ל-0.15 ותראה שהשיא נמוך ומאוחר יותר. זה "השטחת העקומה" שדיברו עליה בתקופת הקורונה.

> 💡 **שני סוגי סימולציה**  
> מודל SIR כפי שכתבנו הוא **דטרמיניסטי**: אין בו הגרלה, ובכל הרצה התוצאה זהה. סימולציית הבדיקה הייתה **סטוכסטית**: היא מבוססת על הגרלות, והתוצאה משתנה מעט בכל הרצה (בלי seed).

## תרגילים

### ✏️ תרגיל 8.1 (קל)

הגרל 10,000 הטלות קובייה ובדוק איזה חלק מהן יצא 6. האם התוצאה קרובה ל-1/6?

</div>

In [ ]:
# תרגיל 8.1: כתוב כאן את הפתרון שלך

In [ ]:
#@title 👀 הצג פתרון לתרגיל 8.1 { display-mode: "form" }
# double-click this cell to see the solution

random.seed(7)
rolls = [random.randint(1, 6) for _ in range(10000)]
print(rolls.count(6) / len(rolls))
print(1 / 6)

# expected output:
# 0.1695
# 0.16666666666666666

<div dir="rtl">

### ✏️ תרגיל 8.2 (בינוני)

הגרל לחץ דם סיסטולי ל-1,000 מטופלים מהתפלגות נורמלית עם ממוצע 125 וסטיית תקן 15. כמה אחוזים מהם עם 140 ומעלה? צייר היסטוגרמה.

</div>

In [ ]:
# תרגיל 8.2: כתוב כאן את הפתרון שלך

In [ ]:
#@title 👀 הצג פתרון לתרגיל 8.2 { display-mode: "form" }
# double-click this cell to see the solution

random.seed(5)
bp = [random.gauss(125, 15) for _ in range(1000)]
high = [b for b in bp if b >= 140]
print(f"{len(high) / len(bp):.1%}")

plt.hist(bp, bins=30, edgecolor="black")
plt.xlabel("Systolic BP (mmHg)")
plt.ylabel("Patients")
plt.show()

# expected output:
# 14.6%
# התוצאה קרובה ל-16%, כי 140 נמצא סטיית תקן אחת מעל הממוצע, ובהתפלגות נורמלית כ-16% מהערכים נמצאים מעל נקודה כזו. ההפרש נובע מהאקראיות של המדגם.

<div dir="rtl">

### ✏️ תרגיל 8.3 (אתגר)

בחדר יש 23 אנשים. מה הסיכוי ששניים מהם נולדו באותו יום בשנה? הרץ 10,000 סימולציות. רמז: הגרל 23 ימים בין 1 ל-365, והשתמש ב-`set` כדי לבדוק אם יש כפילות.

</div>

In [ ]:
# תרגיל 8.3: כתוב כאן את הפתרון שלך

In [ ]:
#@title 👀 הצג פתרון לתרגיל 8.3 { display-mode: "form" }
# double-click this cell to see the solution

random.seed(3)
trials = 10000
same = 0
for _ in range(trials):
    days = [random.randint(1, 365) for _ in range(23)]
    if len(set(days)) < 23:     # a set drops duplicates
        same += 1
print(same / trials)

# expected output:
# 0.5071
# כ-50%, הרבה יותר ממה שרוב האנשים מנחשים. זה "פרדוקס יום ההולדת" המפורסם.

<div dir="rtl">

### ✏️ תרגיל 8.4 (אתגר)

שנה את סימולציית הבדיקה כך שתרוץ בתוך פונקציה `ppv(prevalence, sensitivity, specificity)`. הפעל אותה עם שכיחות 1%, 10% ו-30%. מה קורה לערך הניבוי החיובי?

</div>

In [ ]:
# תרגיל 8.4: כתוב כאן את הפתרון שלך

In [ ]:
#@title 👀 הצג פתרון לתרגיל 8.4 { display-mode: "form" }
# double-click this cell to see the solution

def ppv(prevalence, sensitivity, specificity, n=100_000):
    tp = fp = 0
    for _ in range(n):
        sick = random.random() < prevalence
        positive = random.random() < (sensitivity if sick else 1 - specificity)
        if positive and sick:
            tp += 1
        elif positive:
            fp += 1
    return tp / (tp + fp)

random.seed(0)
for p in [0.01, 0.10, 0.30]:
    print(f"prevalence {p:.0%}: PPV {ppv(p, 0.9, 0.95):.1%}")

# expected output:
# prevalence 1%: PPV 14.6%
# prevalence 10%: PPV 67.6%
# prevalence 30%: PPV 88.4%
# ככל שהמחלה שכיחה יותר, תוצאה חיובית אמינה יותר. לכן בדיקת סקר באוכלוסייה כללית מייצרת הרבה תוצאות חיוביות שגויות, ואותה בדיקה אצל מטופל עם תסמינים אמינה הרבה יותר.

<div dir="rtl">

## ההערות שלי

כתוב כאן מה הבנת, מה לא ברור, ומה לשאול בתרגול. לחיצה כפולה על התא פותחת אותו לעריכה.

-

</div>